# DE2 — Assignment 2: Text — Inverted Index
> Author : Badr TAJINI - Data Engineering II (Data-Intensive Workloads) - ESIEE 2025-2026

**Track:** A

**Names:** *(Student 1 — Student 2)*

We re-use the OpenSky ADS-B CSV dataset.  
The "documents" here are individual aircraft state vectors identified by `icao24`.
We build an inverted index over the **callsign** field (tokenized word-by-word)
so that a user can look up which aircraft ICAOs were associated with any callsign token.

## 0. Setup

In [5]:
import os

os.environ["PYSPARK_PYTHON"] = "/home/maxence/miniconda3/envs/de1-env/bin/python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "/home/maxence/miniconda3/envs/de1-env/bin/python"

In [6]:
import os, time, pathlib
from urllib.parse import urlparse
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType
)

DE2_SPARK_DRIVER_HOST = os.environ.get("DE2_SPARK_DRIVER_HOST", "127.0.0.1")
DE2_SPARK_BIND_ADDRESS = os.environ.get("DE2_SPARK_BIND_ADDRESS", "0.0.0.0")
os.environ.setdefault("SPARK_LOCAL_IP", DE2_SPARK_DRIVER_HOST)


def show_spark_ui(spark_session):
    ui_url = spark_session.sparkContext.uiWebUrl
    print("Spark version:", spark_session.version)
    if ui_url:
        ui_port = urlparse(ui_url).port or 4040
        print("Spark UI:", ui_url)
        print("Spark UI (WSL/Windows browser):", f"http://localhost:{ui_port}")
    else:
        print("Spark UI: not available")


spark = SparkSession.builder \
    .appName("de2-assignment2") \
    .master("local[*]") \
    .config("spark.driver.host", DE2_SPARK_DRIVER_HOST) \
    .config("spark.driver.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .config("spark.ui.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

show_spark_ui(spark)

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_PATH    = "/home/maxence/Documents/DE2_Data_Engineering/assignements/stream_imput"
IDX_PARQUET  = "outputs/lab2/inverted_index"
IDX_CSV      = "outputs/lab2/inverted_index_csv"
PROOF_DIR    = "proof"

for d in [IDX_PARQUET, IDX_CSV, PROOF_DIR]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

Spark version: 4.0.1
Spark UI: http://127.0.0.1:4040
Spark UI (WSL/Windows browser): http://localhost:4040


## 1. Corpus Ingestion

Load the OpenSky CSV with an explicit schema.  
Each row is one state vector; the `icao24` field is the **document ID**
and the `callsign` field is the **text** we index.

In [7]:
schema = StructType([
    StructField("time",          LongType(),   True),
    StructField("icao24",        StringType(), True),
    StructField("lat",           DoubleType(), True),
    StructField("lon",           DoubleType(), True),
    StructField("velocity",      DoubleType(), True),
    StructField("heading",       DoubleType(), True),
    StructField("vertrate",      DoubleType(), True),
    StructField("callsign",      StringType(), True),
    StructField("onground",      StringType(), True),
    StructField("alert",         StringType(), True),
    StructField("spi",           StringType(), True),
    StructField("squawk",        StringType(), True),
    StructField("baroaltitude",  DoubleType(), True),
    StructField("geoaltitude",   DoubleType(), True),
    StructField("lastposupdate", DoubleType(), True),
    StructField("lastcontact",   DoubleType(), True),
])

# Read all CSV files in the directory (handles single file or batch)
corpus = (
    spark.read
         .schema(schema)
         .option("header", "true")
         .csv(DATA_PATH)
)

total_rows = corpus.count()
print(f"Total rows loaded : {total_rows:,}")
print(f"Distinct icao24   : {corpus.select('icao24').distinct().count():,}")
corpus.select("icao24", "callsign", "time").show(10, truncate=False)

Total rows loaded : 1,529,347
Distinct icao24   : 8,495
+------+--------+----------+
|icao24|callsign|time      |
+------+--------+----------+
|4ba9d4|THY82M  |1509962400|
|4853f4|KLM28M  |1509962400|
|406b0d|GOGSE   |1509962400|
|3991e5|AFR1145 |1509962400|
|40703e|GRNJP   |1509962400|
|4b16b9|SWR127H |1509962400|
|4ca242|RYR66PR |1509962400|
|3c66a6|DLH8EJ  |1509962400|
|3c5eee|EWG036  |1509962400|
|394a0a|AFR009  |1509962400|
+------+--------+----------+
only showing top 10 rows


## 2. Text Normalization

Pipeline:
1. Trim whitespace from callsign
2. Lowercase
3. Remove non-alphanumeric characters  
4. Split on whitespace/non-word characters into tokens
5. Explode into one row per token
6. Drop empty tokens and common stop-words (ICAO airline codes are short; we keep tokens ≥ 2 chars)

In [8]:
# Stop-words: very common non-informative tokens in ADS-B callsigns
STOP_WORDS = {"false", "true", "none", "nan", "null", ""}
stop_bcast = spark.sparkContext.broadcast(STOP_WORDS)

from pyspark.sql.functions import udf
from pyspark.sql.types import BooleanType

@udf(BooleanType())
def is_valid_token(token):
    """Return True if token should be kept."""
    if token is None:
        return False
    t = token.strip()
    return len(t) >= 2 and t not in stop_bcast.value

# Count tokens BEFORE stop-word removal
tokens_raw = (
    corpus
    .filter(F.col("callsign").isNotNull())
    .withColumn("callsign_clean", F.trim(F.lower(F.col("callsign"))))
    .withColumn("callsign_clean", F.regexp_replace("callsign_clean", r"[^a-z0-9]", " "))
    .withColumn("token", F.explode(F.split("callsign_clean", r"[\s\W]+")))
)
count_before = tokens_raw.count()

# Apply stop-word filter
tokens_clean = tokens_raw.filter(is_valid_token(F.col("token")))
count_after  = tokens_clean.count()

print(f"Token count BEFORE stop-word removal : {count_before:,}")
print(f"Token count AFTER  stop-word removal : {count_after:,}")
print(f"Tokens dropped                       : {count_before - count_after:,}")

# Keep only the columns we need
doc_tokens = tokens_clean.select(
    F.col("icao24").alias("doc_id"),
    F.col("token")
)
doc_tokens.show(10, truncate=False)

Token count BEFORE stop-word removal : 1,272,933
Token count AFTER  stop-word removal : 1,250,540
Tokens dropped                       : 22,393
+------+-------+
|doc_id|token  |
+------+-------+
|4ba9d4|thy82m |
|4853f4|klm28m |
|406b0d|gogse  |
|3991e5|afr1145|
|40703e|grnjp  |
|4b16b9|swr127h|
|4ca242|ryr66pr|
|3c66a6|dlh8ej |
|3c5eee|ewg036 |
|394a0a|afr009 |
+------+-------+
only showing top 10 rows


## 3. Build Inverted Index

Group by `token`, collect the list of document IDs, and count occurrences.

Schema of the result:
- `token`   — the normalized term  
- `doc_ids` — array of `icao24` values where the token appeared  
- `freq`    — total occurrence count across the corpus

In [9]:
inverted_index = (
    doc_tokens
    .groupBy("token")
    .agg(
        F.collect_list("doc_id").alias("doc_ids"),
        F.count("*").alias("freq")
    )
    .orderBy(F.desc("freq"))
)

unique_terms = inverted_index.count()
print(f"Unique terms in index: {unique_terms:,}")
inverted_index.show(20, truncate=False)

Unique terms in index: 9,553


+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 4. Write to Parquet & CSV

In [10]:
# ── Parquet (compressed, columnar — good for large scale) ────────────────────
inverted_index.write.mode("overwrite").parquet(IDX_PARQUET)
print(f"Parquet index written to {IDX_PARQUET}")

# ── CSV (flat text — easy to inspect, larger on disk) ────────────────────────
# doc_ids is an array; convert to a pipe-separated string for CSV compatibility
index_for_csv = inverted_index.withColumn(
    "doc_ids", F.concat_ws("|", F.col("doc_ids"))
)
index_for_csv.write.mode("overwrite").option("header", "true").csv(IDX_CSV)
print(f"CSV index written to {IDX_CSV}")

Parquet index written to outputs/lab2/inverted_index


CSV index written to outputs/lab2/inverted_index_csv


## 5. Query Latency

Look up specific callsign tokens and measure wall-clock latency for each query,
both on the Parquet index and the in-memory DataFrame.

In [11]:
import time as _time
import csv

QUERY_TERMS = ["ryr", "dlh", "klm"]  # common ICAO airline prefixes in the dataset

# Reload from Parquet to get a cold read
idx_parquet = spark.read.parquet(IDX_PARQUET)

query_results = []
for term in QUERY_TERMS:
    # Query on Parquet
    t0 = _time.time()
    rows_parquet = idx_parquet.filter(F.col("token") == term).collect()
    parquet_ms = round((_time.time() - t0) * 1000, 1)

    # Query on in-memory DataFrame
    t0 = _time.time()
    rows_mem = inverted_index.filter(F.col("token") == term).collect()
    mem_ms = round((_time.time() - t0) * 1000, 1)

    freq = rows_parquet[0]["freq"] if rows_parquet else 0
    print(f"  '{term}': freq={freq}, parquet={parquet_ms} ms, in-memory={mem_ms} ms")
    query_results.append({
        "term": term, "freq": freq,
        "parquet_latency_ms": parquet_ms,
        "inmemory_latency_ms": mem_ms
    })

# Save query plan
plan_query = idx_parquet.filter(F.col("token") == QUERY_TERMS[0]).explain("formatted")
# explain() prints directly; capture via Java explain string
plan_str = idx_parquet.filter(F.col("token") == QUERY_TERMS[0])._jdf.queryExecution().toString()
plan_path = pathlib.Path(PROOF_DIR) / "plan_query.txt"
with open(plan_path, "w") as f:
    f.write(plan_str)
print(f"\nQuery plan saved to {plan_path}")

  'ryr': freq=0, parquet=683.2 ms, in-memory=1774.2 ms


  'dlh': freq=0, parquet=94.1 ms, in-memory=1359.2 ms


  'klm': freq=0, parquet=69.7 ms, in-memory=1674.5 ms
== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [token#392, doc_ids#393, freq#394L]
Batched: true
Location: InMemoryFileIndex [file:/home/maxence/Documents/DE2_Data_Engineering/assignements/outputs/lab2/inverted_index]
PushedFilters: [IsNotNull(token), EqualTo(token,ryr)]
ReadSchema: struct<token:string,doc_ids:array<string>,freq:bigint>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [token#392, doc_ids#393, freq#394L]

(3) Filter [codegen id : 1]
Input [3]: [token#392, doc_ids#393, freq#394L]
Condition : (isnotnull(token#392) AND (token#392 = ryr))



Query plan saved to proof/plan_query.txt


## 6. Footprint Comparison

Compare disk usage of the Parquet index vs the CSV index.

In [12]:
def dir_size_bytes(path_str):
    """Recursively sum file sizes under a directory."""
    total = 0
    for p in pathlib.Path(path_str).rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total

parquet_bytes = dir_size_bytes(IDX_PARQUET)
csv_bytes     = dir_size_bytes(IDX_CSV)

print(f"Parquet index size : {parquet_bytes / 1024:.1f} KB")
print(f"CSV     index size : {csv_bytes / 1024:.1f} KB")
if csv_bytes > 0:
    ratio = csv_bytes / parquet_bytes if parquet_bytes else float('inf')
    print(f"CSV / Parquet ratio: {ratio:.2f}x  (Parquet is {ratio:.1f}x smaller)")

Parquet index size : 163.3 KB
CSV     index size : 8706.1 KB
CSV / Parquet ratio: 53.32x  (Parquet is 53.3x smaller)


## 7. Evidence & Metrics

In [13]:
# ── Save index-build plan ────────────────────────────────────────────────────
build_plan = inverted_index._jdf.queryExecution().toString()
with open(pathlib.Path(PROOF_DIR) / "plan_index_build.txt", "w") as f:
    f.write(build_plan)
print("Index build plan saved.")

# ── Write metrics log ────────────────────────────────────────────────────────
metrics_fields = [
    "metric_type", "description",
    "parquet_value", "csv_value", "unit"
]
metrics_rows = [
    {
        "metric_type" : "footprint",
        "description" : "index disk size",
        "parquet_value": round(parquet_bytes / 1024, 1),
        "csv_value"   : round(csv_bytes / 1024, 1),
        "unit"        : "KB"
    },
    {
        "metric_type" : "latency",
        "description" : f"query '{QUERY_TERMS[0]}'",
        "parquet_value": query_results[0]["parquet_latency_ms"] if query_results else 0,
        "csv_value"   : "N/A",
        "unit"        : "ms"
    },
    {
        "metric_type" : "latency",
        "description" : f"query '{QUERY_TERMS[1]}'",
        "parquet_value": query_results[1]["parquet_latency_ms"] if len(query_results) > 1 else 0,
        "csv_value"   : "N/A",
        "unit"        : "ms"
    },
    {
        "metric_type" : "latency",
        "description" : f"query '{QUERY_TERMS[2]}'",
        "parquet_value": query_results[2]["parquet_latency_ms"] if len(query_results) > 2 else 0,
        "csv_value"   : "N/A",
        "unit"        : "ms"
    },
    {
        "metric_type" : "corpus",
        "description" : "total rows",
        "parquet_value": total_rows,
        "csv_value"   : total_rows,
        "unit"        : "rows"
    },
    {
        "metric_type" : "corpus",
        "description" : "unique terms",
        "parquet_value": unique_terms,
        "csv_value"   : unique_terms,
        "unit"        : "terms"
    },
]

with open("lab2_metrics_log.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=metrics_fields)
    writer.writeheader()
    writer.writerows(metrics_rows)

print("lab2_metrics_log.csv written.")
with open("lab2_metrics_log.csv") as f:
    print(f.read())

Index build plan saved.
lab2_metrics_log.csv written.
metric_type,description,parquet_value,csv_value,unit
footprint,index disk size,163.3,8706.1,KB
latency,query 'ryr',683.2,N/A,ms
latency,query 'dlh',94.1,N/A,ms
latency,query 'klm',69.7,N/A,ms
corpus,total rows,1529347,1529347,rows
corpus,unique terms,9553,9553,terms



## 8. Cleanup

In [4]:
spark.stop()
print("Done.")

Done.
